# Qwen3 0.6B Checkpoint: 200 Test Sample Eval

Loads the base `Qwen/Qwen3-0.6B` model plus a LoRA adapter/checkpoint, rebuilds the same test split used by the sweep, scores 200 test emails with next-token `ham`/`spam` classification, and reports how many were correct.

Edit `CHECKPOINT_PATH` in the first code cell if you want a different checkpoint.

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "dataset").is_dir() and (candidate / "lora-fine-tuning").is_dir():
            return candidate
    raise RuntimeError("Could not find project root containing dataset/ and lora-fine-tuning/.")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

SWEEP_SCRIPT = PROJECT_ROOT / "lora-fine-tuning" / "qwen3_0.6b_casual_lm_sweep.py"
spec = importlib.util.spec_from_file_location("qwen3_sweep_helpers", SWEEP_SCRIPT)
sweep = importlib.util.module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = sweep  # dataclasses expects the module to be registered during import.
spec.loader.exec_module(sweep)

MODEL_ID = sweep.MODEL_ID

# Default: best sweep run from qwen3_clm_sweep_20260506_233813.
# You can also point this at trainer_output/checkpoint-1614 or any folder with adapter_config.json.
CHECKPOINT_PATH = PROJECT_ROOT / "results" / "qwen3_clm_sweep_20260506_233813" / "06_lr_context_seq1024_lr1e-4_r16_a32_do0p1" / "adapter"

TEST_SAMPLE_COUNT = 500
SHUFFLE_TEST = True
SAMPLE_SEED = sweep.SEED
EVAL_BATCH_SIZE = 2  # Keep conservative for Mac/MPS memory. Increase if your Mac is comfortable.

def infer_max_seq_length(checkpoint_path: Path, fallback: int = 1024) -> int:
    for candidate in [checkpoint_path, *checkpoint_path.parents]:
        metrics_path = candidate / "metrics.json"
        config_path = candidate / "config.json"
        if metrics_path.exists():
            with metrics_path.open("r", encoding="utf-8") as f:
                payload = json.load(f)
            return int(payload.get("config", {}).get("max_seq_length", fallback))
        if config_path.exists():
            with config_path.open("r", encoding="utf-8") as f:
                payload = json.load(f)
            return int(payload.get("config", {}).get("max_seq_length", fallback))
    return fallback

CHECKPOINT_PATH = Path(CHECKPOINT_PATH).expanduser().resolve()
MAX_SEQ_LENGTH = infer_max_seq_length(CHECKPOINT_PATH)

if not (CHECKPOINT_PATH / "adapter_config.json").exists():
    raise FileNotFoundError(f"No adapter_config.json found in {CHECKPOINT_PATH}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Model:        {MODEL_ID}")
print(f"Checkpoint:   {CHECKPOINT_PATH}")
print(f"Max seq len:  {MAX_SEQ_LENGTH}")

Project root: /Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska
Model:        Qwen/Qwen3-0.6B
Checkpoint:   /Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/results/qwen3_clm_sweep_20260506_233813/06_lr_context_seq1024_lr1e-4_r16_a32_do0p1/adapter
Max seq len:  1024


In [2]:
from datasets import ClassLabel, load_dataset
from transformers import AutoTokenizer
from dataset.combine import combine_datasets

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

label_token_id_lists = {
    sweep.NEGATIVE_LABEL_TEXT: sweep.encode_text(tokenizer, sweep.NEGATIVE_LABEL_TEXT),
    sweep.POSITIVE_LABEL_TEXT: sweep.encode_text(tokenizer, sweep.POSITIVE_LABEL_TEXT),
}
for label_text, token_ids in label_token_id_lists.items():
    if len(token_ids) != 1:
        raise ValueError(f"Label {label_text!r} is not a single token: {token_ids}")
label_token_ids = {label_text: token_ids[0] for label_text, token_ids in label_token_id_lists.items()}

data_path = combine_datasets(["spam_assassin"], spam_ham_ratio=0.5)
raw_dataset = load_dataset("parquet", data_files=str(data_path), split="train")
raw_dataset = raw_dataset.cast_column("label", ClassLabel(names=["valid", "spam"]))

holdout = raw_dataset.train_test_split(
    test_size=sweep.HOLDOUT_SPLIT,
    stratify_by_column="label",
    seed=sweep.SEED,
)
valid_test = holdout["test"].train_test_split(
    test_size=sweep.TEST_SPLIT / sweep.HOLDOUT_SPLIT,
    stratify_by_column="label",
    seed=sweep.SEED,
)

test_dataset = valid_test["test"].filter(
    lambda sample: bool(sweep.build_email_text(sample["subject"], sample["body"])),
    desc="Filtering empty emails",
)
if SHUFFLE_TEST:
    test_dataset = test_dataset.shuffle(seed=SAMPLE_SEED)
test_dataset = test_dataset.select(range(min(TEST_SAMPLE_COUNT, len(test_dataset))))

test_dataset = test_dataset.map(
    lambda sample: sweep.build_tokenized_sample(tokenizer, sample, MAX_SEQ_LENGTH),
    desc=f"Formatting 200 test prompts to <= {MAX_SEQ_LENGTH} tokens",
)

trimmed_count = sum(bool(value) for value in test_dataset["was_trimmed"])
print(f"Dataset path:  {data_path}")
print(f"Test samples:  {len(test_dataset)}")
print(f"Trimmed rows:  {trimmed_count}/{len(test_dataset)}")
print(f"Label tokens:  {label_token_ids}")

Combined dataset already exists: /Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/dataset/combined_datasets/generated/spam_assassin__dedupe_high__spam_0_5__495c20e471.parquet


Filtering empty emails:   0%|          | 0/120 [00:00<?, ? examples/s]

Formatting 200 test prompts to <= 1024 tokens:   0%|          | 0/120 [00:00<?, ? examples/s]

Dataset path:  /Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/dataset/combined_datasets/generated/spam_assassin__dedupe_high__spam_0_5__495c20e471.parquet
Test samples:  120
Trimmed rows:  37/120
Label tokens:  {'ham': 5604, 'spam': 75545}


In [3]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

dtype = torch.float16 if device.type in {"mps", "cuda"} else torch.float32
print(f"Using device: {device}; dtype: {dtype}")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)
model = PeftModel.from_pretrained(base_model, str(CHECKPOINT_PATH))
model.to(device)
model.eval()

print("Loaded base model + LoRA adapter.")

Using device: mps; dtype: torch.float16


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loaded base model + LoRA adapter.


In [4]:
# Paste any email text here, then run this cell after the model-loading cell.
import pandas as pd

CUSTOM_EMAIL_TEXT = """
Subject: Last meating with claudia.

Hi Josh,
You will not believe it. After our meeting with claudia I clapped her cheeks and secured us a great deal. That's what I call killing two birds with one stone.
On another note, please let me know if you are available for a meeting next Tuesday.

Cheers,
Brad


""".strip()

def score_custom_email(email_text: str):
    if not email_text.strip():
        raise ValueError("CUSTOM_EMAIL_TEXT is empty.")

    trimmed = sweep.trim_email_to_fit(
        tokenizer=tokenizer,
        email_text=email_text,
        completion_text=f"{sweep.POSITIVE_LABEL_TEXT}{sweep.IM_END_TOKEN}",
        max_seq_length=MAX_SEQ_LENGTH,
    )
    prompt_ids = trimmed["prompt_ids"]
    label_names = [sweep.NEGATIVE_LABEL_TEXT, sweep.POSITIVE_LABEL_TEXT]
    label_ids = torch.tensor(
        [label_token_ids[sweep.NEGATIVE_LABEL_TEXT], label_token_ids[sweep.POSITIVE_LABEL_TEXT]],
        dtype=torch.long,
        device=device,
    )

    with torch.inference_mode():
        batch = sweep.pad_prompt_batch(torch, [prompt_ids], tokenizer.pad_token_id, device)
        outputs = model(**batch)
        last_position = batch["attention_mask"].sum(dim=1) - 1
        next_token_logits = outputs.logits[torch.arange(1, device=device), last_position]
        label_logits = next_token_logits.index_select(dim=-1, index=label_ids)
        probabilities = torch.softmax(torch.nan_to_num(label_logits), dim=-1)[0].detach().cpu().float().tolist()

    prediction_index = int(probabilities[1] >= probabilities[0])
    return {
        "prediction": label_names[prediction_index],
        "p_ham": probabilities[0],
        "p_spam": probabilities[1],
        "prompt_token_length": len(prompt_ids),
        "raw_email_tokens": trimmed["raw_email_tokens"],
        "trimmed_email_tokens": trimmed["trimmed_email_tokens"],
        "was_trimmed": trimmed["was_trimmed"],
    }

custom_result = score_custom_email(CUSTOM_EMAIL_TEXT)
display(pd.DataFrame([custom_result]))
print(f"Prediction: {custom_result['prediction']} | ham={custom_result['p_ham']:.4f} spam={custom_result['p_spam']:.4f}")

,prediction,p_ham,p_spam,prompt_token_length,raw_email_tokens,trimmed_email_tokens,was_trimmed
0,spam,0.313965,0.686035,112,71,71,False


Prediction: spam | ham=0.3140 spam=0.6860


In [5]:
import pandas as pd
from tqdm.auto import tqdm

def predict_next_token(dataset_split, batch_size: int = EVAL_BATCH_SIZE):
    model.eval()
    predictions = []
    labels = []
    probabilities = []
    records = []
    failure_count = 0
    label_names = [sweep.NEGATIVE_LABEL_TEXT, sweep.POSITIVE_LABEL_TEXT]
    label_ids = torch.tensor(
        [label_token_ids[sweep.NEGATIVE_LABEL_TEXT], label_token_ids[sweep.POSITIVE_LABEL_TEXT]],
        dtype=torch.long,
        device=device,
    )

    with torch.inference_mode():
        for start in tqdm(range(0, len(dataset_split), batch_size)):
            end = min(start + batch_size, len(dataset_split))
            rows = dataset_split.select(range(start, end))
            prompt_input_ids = rows["prompt_input_ids"]
            batch = sweep.pad_prompt_batch(torch, prompt_input_ids, tokenizer.pad_token_id, device)
            outputs = model(**batch)
            last_positions = batch["attention_mask"].sum(dim=1) - 1
            next_token_logits = outputs.logits[torch.arange(len(prompt_input_ids), device=device), last_positions]
            label_logits = next_token_logits.index_select(dim=-1, index=label_ids)
            finite = torch.isfinite(label_logits).all(dim=-1)
            safe_logits = torch.nan_to_num(label_logits, nan=-1e9, posinf=1e9, neginf=-1e9)
            batch_probabilities = torch.softmax(safe_logits, dim=-1)
            batch_predictions = batch_probabilities.argmax(dim=-1)

            failure_count += int((~finite).sum().item())
            batch_probabilities = batch_probabilities.detach().cpu().float().tolist()
            batch_predictions = batch_predictions.detach().cpu().int().tolist()

            for offset, row in enumerate(rows):
                true_label = int(row["label"])
                pred_label = int(batch_predictions[offset])
                prob_ham, prob_spam = batch_probabilities[offset]
                predictions.append(pred_label)
                labels.append(true_label)
                probabilities.append([prob_ham, prob_spam])
                records.append(
                    {
                        "sample_index": start + offset,
                        "actual": label_names[true_label],
                        "predicted": label_names[pred_label],
                        "correct": pred_label == true_label,
                        "p_ham": prob_ham,
                        "p_spam": prob_spam,
                        "subject": row.get("subject") or "",
                        "was_trimmed": bool(row["was_trimmed"]),
                        "token_length": int(row["token_length"]),
                    }
                )

    metrics = sweep.compute_metrics(predictions, labels, probabilities, failure_count)
    return metrics, pd.DataFrame(records)

metrics, predictions_df = predict_next_token(test_dataset)
correct = int(predictions_df["correct"].sum())
total = len(predictions_df)

summary = pd.DataFrame(
    [
        {"metric": "correct", "value": correct},
        {"metric": "total", "value": total},
        {"metric": "accuracy", "value": metrics["accuracy"]},
        {"metric": "f1", "value": metrics["f1"]},
        {"metric": "precision", "value": metrics["precision"]},
        {"metric": "recall", "value": metrics["recall"]},
        {"metric": "specificity", "value": metrics["specificity"]},
        {"metric": "false_positive_count", "value": metrics["false_positive_count"]},
        {"metric": "false_negative_count", "value": metrics["false_negative_count"]},
        {"metric": "classification_failure_count", "value": metrics["classification_failure_count"]},
    ]
)

print(f"Got {correct}/{total} correct ({metrics['accuracy']:.2%}).")
display(summary)
display(pd.crosstab(predictions_df["actual"], predictions_df["predicted"], rownames=["actual"], colnames=["predicted"], dropna=False))

  0%|          | 0/60 [00:00<?, ?it/s]

Got 109/120 correct (90.83%).


,metric,value
0,correct,109.000000
1,total,120.000000
2,accuracy,0.908333
3,f1,0.909091
4,precision,0.901639
5,recall,0.916667
6,specificity,0.900000
7,false_positive_count,6.000000
8,false_negative_count,5.000000
9,classification_failure_count,0.000000


predicted,ham,spam
actual,,
ham,54,6
spam,5,55


In [6]:
mistakes_df = predictions_df.loc[~predictions_df["correct"]].copy()
display(predictions_df.head(20))
print(f"Mistakes: {len(mistakes_df)}")
display(mistakes_df)

output_path = PROJECT_ROOT / "results" / "qwen3_0.6b_checkpoint_200_test_predictions.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
predictions_df.to_csv(output_path, index=False)
print(f"Saved per-sample predictions to {output_path}")

,sample_index,actual,predicted,correct,p_ham,p_spam,subject,was_trimmed,token_length
0,0,ham,ham,True,1.000000,0.000041,[SAdev] [Bug 1054] New: Split up FROM_ENDS_IN_...,True,1024
1,1,ham,ham,True,0.999512,0.000417,[ILUG] cups question Sender: ilug-admin@linux....,False,542
2,2,spam,spam,True,0.035156,0.964844,YOUR ACCOUNT HAS BEEN CLOSED! Sender: Sportspi...,False,321
3,3,spam,spam,True,0.000017,1.000000,Order your Viagra and weight-loss here 6117kFv...,False,982
4,4,spam,spam,True,0.001649,0.998535,I don't work...but I have a ton of money! vhcw...,False,957
5,5,spam,spam,True,0.000817,0.999023,Reach millions on the internet!! Dear Consumer...,True,1023
6,6,spam,spam,True,0.000374,0.999512,Immediate Reply Needed,False,650
7,7,spam,spam,True,0.204346,0.795898,CDR: Kime Oy Vereceksiniz ?,False,976
8,8,spam,spam,True,0.191895,0.808105,[dcms-dev] MY INHERITANCE Sender: dcms-dev-adm...,False,798
9,9,ham,ham,True,0.999512,0.000330,Re: [ILUG] Cobalt question,False,374


Mistakes: 11


,sample_index,actual,predicted,correct,p_ham,p_spam,subject,was_trimmed,token_length
14,14,spam,ham,False,0.936523,0.063721,10 million fresh email addresses sent to you o...,True,1024
16,16,ham,spam,False,0.004398,0.995605,Black entrepreneurs 'face bank bias',False,134
23,23,ham,spam,False,0.000744,0.999023,Mass human sacrifice unearthed in Peru,False,113
28,28,spam,ham,False,0.972656,0.027588,CDR: Get McAfee VirusScan for just $19.95! 60%...,True,1023
49,49,ham,spam,False,0.000405,0.999512,"Toddler falls from a first-storey window, save...",False,144
66,66,ham,spam,False,0.000897,0.999023,Executive pay leaps ahead 17%,False,132
76,76,spam,ham,False,0.792969,0.206909,=?Big5?B?s8y3c6V4xles2aR1sNOmV7/9LTEtMTY3LQ==?=,True,1024
81,81,ham,spam,False,0.000296,0.999512,Female circumcision does not reduce sexual act...,False,108
85,85,ham,spam,False,0.002846,0.997070,"Forged documents, public drinking, massive fis...",False,131
89,89,spam,ham,False,0.985840,0.014061,[ILUG] Re: whats up -colonize Sender: ilug-adm...,False,390


Saved per-sample predictions to /Users/wojciechdrozdz/uni/Magisterka/Semestr III/Praca Magisterska/results/qwen3_0.6b_checkpoint_200_test_predictions.csv
